In [32]:
print("hallow world")

hallow world


In [33]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import datetime, timedelta, time
import os
import os
import sys
import pandas as pd
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings("ignore")
import plotly.express as px

In [42]:
dataset_path = r'C:\Users\LENOVO\MachineLearningProjects\AutonomousDataAnalystAgent\data\raw_path\x_data.xlsx'

In [43]:
new_df = pd.read_excel(r'C:\Users\LENOVO\MachineLearningProjects\AutonomousDataAnalystAgent\data\raw_path\x_data.xlsx')

In [44]:
new_df.shape

(32873, 34)

In [46]:
def get_remarks_df(df):
    """
    Process DataFrame to extract and flag specific patterns from REMARKS column.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input DataFrame containing complaint/tweet data
        
    Returns:
    --------
    pd.DataFrame
        Filtered DataFrame with extracted features and flags
    """
    df = df.copy()
    
    # ========================================================================
    # DUPLICATE CASE: Extract all 3-5 digit numbers from REMARKS
    # ========================================================================
    df['Duplicate Case'] = (
        df['REMARKS']
        .astype(str)
        .str.findall(r'(?i)\b\d{3,5}\b')
    )
    
    # Join list of matches into a single string (comma-separated)
    df['Duplicate Case'] = (
        df['Duplicate Case']
        .apply(lambda x: ', '.join(x) if x else '')
        .replace('', 'NA')
        .fillna('NA')
    )
    
    # ========================================================================
    # APPRECIATION TWEET: Extract and normalize mentions
    # ========================================================================
    df['Appreciation Tweet'] = (
        df['REMARKS']
        .astype(str)
        .str.findall(r'(?i)\bappreciation\s*tweet\b')
        .apply(lambda x: 'Appreciation Tweet' if x else 'NA')
    )
    
    # ========================================================================
    # AWAITED CONSUMER ID: Flag mentions of awaited consumer/details/response
    # ========================================================================
    df['Awaited Consumer Id'] = (
        df['REMARKS']
        .astype(str)
        .str.contains(
            r'(?i)\b(awaited\s*consumer|details?|exacts?|response?s?|id|asked|ask)\b',
            regex=True,
            na=False
        )
        .map({True: 'Yes', False: 'No'})
        .fillna('NA')
    )
    
    # ========================================================================
    # X USER ID: Extract Twitter/X username from URL
    # ========================================================================
    df['X User Id'] = (
        df['TWEET/LINK']
        .astype(str)
        .str.extract(r'(?:twitter\.com|x\.com)/([^/]+)/', expand=False)
    )
    
    df['X User Id'] = (
        df['X User Id']
        .apply(lambda x: '@' + x if pd.notna(x) and x != 'nan' and x else 'NA')
    )
    
    # ========================================================================
    # STREETLIGHT AND NOT TPWODL: Flag streetlight issues outside TPWODL
    # jurisdiction or containing location/jurisdiction keywords
    # ========================================================================
    df['Streetlight And Not TPWODL'] = (
        (
            df['REMARKS']
            .astype(str)
            .str.contains(r'(?i)\bstreetlight(s)?\b', regex=True, na=False) &
            ~df['REMARKS']
            .astype(str)
            .str.contains(r'(?i)\bTPWODL\b', regex=True, na=False)
        )
        |
        df['REMARKS']
        .astype(str)
        .str.contains(
            r'(?i)\b(not\s+related|dist|jurisdiction|jurisdictions|districts?|'
            r'not\s+under|not\s+coming|under|bhubaneswar|khordha|balasore|city|'
            r'angul|dhenkanal|boudh|koraput|nayagarh|ganjam|mayurbhanj|'
            r'nabarangpur|kandhamal|cuttack)\b',
            regex=True,
            na=False
        )
    )
    
    df['Streetlight And Not TPWODL'] = (
        df['Streetlight And Not TPWODL']
        .map({True: 'Yes', False: 'No'})
        .fillna('No')
    )
    
    # ========================================================================
    # CALL INITIATED CONSUMERS: Flag mentions of callback done
    # ========================================================================
    df['Call Initiated Consumers'] = (
        df['REMARKS']
        .astype(str)
        .str.contains(
            r'(?i)\b(call\s*back\s*done|callback\s*done|callcack\s*done)\b',
            regex=True,
            na=False
        )
        .map({True: 'Yes', False: 'No'})
        .fillna('NA')
    )
    
    # ========================================================================
    # ELEPHANT MOVEMENT: Flag mentions of elephant-related issues
    # ========================================================================
    df['Elephant Movement'] = (
        df['REMARKS']
        .astype(str)
        .str.contains(
            r'(?i)\b(elephant\s*movement|elephan\s*movement|movementum|elephant)\b',
            regex=True,
            na=False
        )
        .map({True: 'Yes', False: 'No'})
        .fillna('NA')
    )
    
    # ========================================================================
    # MAILED SENT: Flag mentions of email/mail communication
    # ========================================================================
    df['Mailed Sent'] = (
        df['REMARKS']
        .astype(str)
        .str.contains(
            r'(?i)\b(mailed|mail|mails|email|emailing)\b',
            regex=True,
            na=False
        )
        .map({True: 'Yes', False: 'No'})
        .fillna('NA')
    )
    
    # ========================================================================
    # SELECT AND RETURN SPECIFIC COLUMNS
    # ========================================================================
    remarks_filter_df = df[[
        'SL.NO',
        'DATE',
        'DIVISION',
        'CIRCLE',
        'COMPLAINT TYPE',
        'REMARKS',
        'TWEET/LINK',
        'COMPLAINANT NAME',
        'Duplicate Case',
        'Appreciation Tweet',
        'Awaited Consumer Id',
        'X User Id',
        'Streetlight And Not TPWODL',
        'Call Initiated Consumers',
        'Elephant Movement',
        'Mailed Sent'
    ]]
    
    return remarks_filter_df

In [47]:
remarks_df = get_remarks_df(new_df)

In [48]:
remarks_df.columns

Index(['SL.NO', 'DATE', 'DIVISION', 'CIRCLE', 'COMPLAINT TYPE', 'REMARKS',
       'TWEET/LINK', 'COMPLAINANT NAME', 'Duplicate Case',
       'Appreciation Tweet', 'Awaited Consumer Id', 'X User Id',
       'Streetlight And Not TPWODL', 'Call Initiated Consumers',
       'Elephant Movement', 'Mailed Sent'],
      dtype='object')

In [51]:


# Check value counts for all new columns
print("=" * 50)
print("VALUE COUNTS FOR ALL COLUMNS")
print("=" * 50)

columns_to_check = [
    'Duplicate Case', 
    'Appreciation Tweet', 
    'Awaited Consumer Id',
    'X User Id', 
    'Streetlight And Not TPWODL',
    'Call Initiated Consumers', 
    'Elephant Movement', 
    'Mailed Sent'
]

for col in columns_to_check:
    print(f"\n{col}:")
    print(remarks_df[col].value_counts())
    print("-" * 50)

VALUE COUNTS FOR ALL COLUMNS

Duplicate Case:
Duplicate Case
NA                   26295
8505                    27
18325, 6920             27
17794, 6287, 2018       24
8724                    20
                     ...  
32862                    1
1468                     1
600                      1
2023, 2024               1
612                      1
Name: count, Length: 2923, dtype: int64
--------------------------------------------------

Appreciation Tweet:
Appreciation Tweet
NA                    29429
Appreciation Tweet     3444
Name: count, dtype: int64
--------------------------------------------------

Awaited Consumer Id:
Awaited Consumer Id
No     27632
Yes     5241
Name: count, dtype: int64
--------------------------------------------------

X User Id:
X User Id
NA                  7593
@rkb97official       593
@Bikramkesh4707      453
@akshya_patra        433
@CharubalaB          364
                    ... 
@Kanheimishra          1
@swarupkumarmis3       1
@bajrangsul